# Generate the descriptive FD001 t-SNE figure.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
PROJECT_DIR = PROJECT_ROOT


In [ ]:
import subprocess,sys,os,time,json,platform
from pathlib import Path
from datetime import datetime,timezone


def python_candidates(explicit=None):
    if explicit:
        return [str(Path(explicit).expanduser().absolute())]
    candidates=[Path(sys.executable)]
    bases={Path(sys.prefix),Path(sys.base_prefix),Path(sys.executable).parent.parent,
           Path('/opt/anaconda3'),Path('/opt/miniconda3'),Path.home()/'miniconda3',Path.home()/'anaconda3'}
    for base in sorted(bases):
        candidates.extend(sorted((base/'envs').glob('*/bin/python')))
    candidates.extend(sorted((Path.home()/'.conda/envs').glob('*/bin/python')))
    registry=Path.home()/'.conda/environments.txt'
    if registry.is_file():
        candidates.extend(Path(line.strip())/'bin/python' for line in registry.read_text().splitlines() if line.strip())
    for base in [Path.cwd(),Path.cwd().parent]:
        candidates.extend([base/'.venv/bin/python',base/'venv/bin/python'])

    unique=[]
    for path in candidates:
        value=str(path.expanduser().absolute())
        if path.is_file() and value not in unique:unique.append(value)
    return unique


def compatible_python(expected,out,explicit=None):
    code="""import json,platform,importlib.metadata as m
v={'python':platform.python_version()}
for key,package in [('numpy','numpy'),('pandas','pandas'),('sklearn','scikit-learn'),('tensorflow','tensorflow')]:
 try:v[key]=m.version(package)
 except m.PackageNotFoundError:v[key]='NOT_INSTALLED'
print(json.dumps(v))
"""
    reports=[]
    for executable in python_candidates(explicit):
        item={'executable':executable}
        try:
            proc=subprocess.run([executable,'-c',code],capture_output=True,text=True,timeout=20)
            item['metadata_returncode']=proc.returncode
            if proc.returncode!=0:
                item['status']='METADATA_FAILED';item['stderr_tail']=proc.stderr[-1500:];reports.append(item);continue
            versions=json.loads(proc.stdout.strip().splitlines()[-1]);item['versions']=versions
            if any(versions.get(k)!=v for k,v in expected.items()):
                item['status']='VERSIONS_DIFFER';reports.append(item);continue
            print('Eğitim sürümleriyle eşleşen Python kontrol ediliyor:',executable,flush=True)
            probe="import numpy, pandas; import tensorflow as tf; import sklearn; print('TF_PROBE_OK',float(tf.reduce_sum(tf.constant([1.,2.])).numpy()),flush=True)"
            proc=subprocess.run([executable,'-X','faulthandler','-c',probe],capture_output=True,text=True,timeout=60)
            item.update(probe_returncode=proc.returncode,probe_stdout_tail=proc.stdout[-1500:],probe_stderr_tail=proc.stderr[-3000:])
            if proc.returncode==0 and 'TF_PROBE_OK 3.0' in proc.stdout:
                item['status']='READY';reports.append(item);return executable,reports
            item['status']='IMPORT_OR_TENSOR_TEST_FAILED'
        except subprocess.TimeoutExpired:
            item['status']='PROBE_TIMEOUT'
        except (OSError,ValueError) as e:
            item.update(status='PROBE_ERROR',error=str(e))
        reports.append(item)
    return None,reports




CONFIG={'seed': 42, 'fold': 1, 'fold_relative_directory': '05_FD001_grouped_cv/20261027T085136_896849Z/seed_42/fold_01', 'known_hashes': {'02_FD001_protocol/engine_splits.csv': '53c2ed564fcded8e502f307c0011aa1cd12ebd03b4429882e1fdf142466f7c45', '02_FD001_protocol/train_evaluation_reference.csv': 'c6b5e08384280fb687b188723fa7ecf7e76254a9ec20fc606be206f8de7a27e8', '05_FD001_grouped_cv/20261027T085136_896849Z/seed_42/fold_01/encoder.keras': '0755bc949327dab89f1ee90696e35679191fa432d5d984ed4459a0c7be6785a1', '05_FD001_grouped_cv/20261027T085136_896849Z/seed_42/fold_01/scaler_parameters.json': 'a5f23365e0ef431845766d0f439e6850e80a35c1435ddcb65c12d3f32540db61'}, 'train_sha256': '963b5e22825b34d8b21c69e1aeb4af3e647050eb672ee8834ba4b5d91d2de0f8', 'expected_versions': {'python': '3.12.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'tensorflow': '2.21.0'}}
WORKER = load_text('assets/13/02_worker.txt')


In [ ]:
import tempfile
import traceback
out = PROJECT_DIR/'revision_outputs'/'13_FD001_figure7_tsne'
out.mkdir(parents=True,exist_ok=True)
with tempfile.TemporaryDirectory(dir=out) as name:
    work=Path(name)
    try:
        exe,checks=compatible_python(CONFIG['expected_versions'],work,None)
        if not exe:
            raise RuntimeError('Uyumlu Python bulunamadı: '+str(checks))
        (work/'worker.py').write_text(WORKER)
        (work/'config.json').write_text(json.dumps(dict(CONFIG,project=str(PROJECT_DIR))))
        with (work/'worker.log').open('w') as log:
            proc=subprocess.run([exe,'-u','-X','faulthandler',str(work/'worker.py'),str(work/'config.json'),str(work/'result.json')],stdout=log,stderr=subprocess.STDOUT)
        if proc.returncode:
            raise RuntimeError((work/'worker.log').read_text()[-5000:])
        result=json.loads((work/'result.json').read_text())
    except Exception:
        result=dict(status='REVIEW_REQUIRED',error=traceback.format_exc())
    summary=out/'figure7_tsne_summary.json'
    tmp=work/'final.json'
    tmp.write_text(json.dumps(result,ensure_ascii=False,allow_nan=False))
    tmp.replace(summary)
print('DURUM:',result['status'])
if 'error' in result: print(result['error'])
print('GÖNDERİLECEK DOSYA:',summary)
